# 🎬 AI Cartoon Video Generator
**Generate YouTube-quality cartoon story videos - 100% FREE**

### Steps:
1. **Runtime > Change runtime type > T4 GPU** (important!)
2. Run Cell 1 (install ~3 min)
3. Edit your story in Cell 2
4. Run Cell 2 (generate ~8-12 min)
5. Run Cell 3 (watch & download)

In [ ]:
#@title **CELL 1: Install Everything (run once)** { display-mode: "form" }

!pip install -q diffusers==0.25.0 transformers accelerate torch safetensors
!pip install -q xformers
!pip install -q Pillow requests pydantic
!pip install -q piper-tts
!apt-get install -qq ffmpeg > /dev/null 2>&1

# Clone latest code
!rm -rf ai-generator 2>/dev/null
!git clone https://github.com/Khushalpatel499/ai-generator.git
%cd ai-generator

# Download TTS voice model
!mkdir -p models
!wget -q -O models/en_US-lessac-medium.onnx https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx
!wget -q -O models/en_US-lessac-medium.onnx.json https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx.json

# Verify GPU
import torch
print("\n" + "="*50)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("READY! Go to Cell 2.")
else:
    print("NO GPU! Go to Runtime > Change runtime type > T4 GPU")
print("="*50)

In [ ]:
#@title **CELL 2: Generate Video** { display-mode: "form" }

#@markdown ### Settings
STYLE = "cartoon" #@param ["cartoon", "anime", "pixar", "comic"]
ENABLE_SUBTITLES = True #@param {type:"boolean"}
SD_STEPS = 28 #@param {type:"slider", min:15, max:50, step:1}

#@markdown ### Story (edit below)

STORY = """
In a peaceful village deep inside a green valley, animals lived happily together. Among them was a small orange cat named Milo, who was often ignored because he was tiny and clumsy.

Other animals would say, "He is too small to help us." But Milo never stopped trying to prove himself.

One day, the sky turned dark. Strong winds started blowing. The Elder Owl announced, "A great storm is coming, stronger than anything we have seen before."

All animals panicked and ran for shelter. But Milo noticed something strange. The storm was not natural. A dark swirling cloud with glowing red eyes was moving toward the village. It was the Storm Spirit.

The Storm Spirit roared, "I will destroy everything!" Lightning crashed all around. The village was in chaos.

But Milo did not run. He climbed the highest hill, facing the storm alone. "I may be small, but I will not run away!" he shouted into the wind.

At the top of the hill, Milo found an ancient magical bell. He rang it with all his strength. A brilliant golden light burst from the bell, spreading across the sky.

The Storm Spirit screamed as the light dissolved its darkness. "Nooo!" it cried, shrinking smaller and smaller until it vanished completely.

The sky cleared. Sunshine returned. All the village animals gathered around Milo, cheering. The Elder Owl smiled and said, "True courage comes from the heart, not from size."

From that day on, Milo was known as the bravest hero in the valley. And he proved that even the smallest can become the greatest.
"""

# ====== PIPELINE (don't edit below) ======
import sys
sys.path.insert(0, '.')

from src.core.config import Config
from src.pipeline.orchestrator import Pipeline, Job
from src.modules.scene.pro_generator import ProSceneGenerator
from src.modules.image.generator import StableDiffusionGenerator
from src.modules.tts.engine import PiperTTSEngine
from src.modules.animation.animator import FFmpegAnimator
from src.modules.video.composer import FFmpegComposer
from src.modules.video.subtitles import SubtitleGenerator
from src.modules.storage.local import LocalStorage

config = Config()
config.sd_steps = SD_STEPS
config.sd_width = 1024
config.sd_height = 576
config.video_fps = 24
config.transition_duration = 1.0
config.ensure_dirs()

pipeline = Pipeline(
    scene_gen=ProSceneGenerator(),
    image_gen=StableDiffusionGenerator(config),
    tts_engine=PiperTTSEngine(config),
    animator=FFmpegAnimator(config),
    composer=FFmpegComposer(config),
    storage=LocalStorage(config),
    subtitle_gen=SubtitleGenerator() if ENABLE_SUBTITLES else None,
)

job = Job(story=STORY, style=STYLE, subtitles=ENABLE_SUBTITLES, background_music=False)

print(f"Generating video [{job.id}]")
print(f"Style: {STYLE} | Steps: {SD_STEPS} | Subtitles: {ENABLE_SUBTITLES}")
print(f"This takes ~8-12 minutes on T4 GPU...")
print()

import time
start = time.time()
result = pipeline.run(job)
elapsed = time.time() - start

print(f"\n{'='*50}")
if result.status.value == "completed":
    print(f"VIDEO READY! ({elapsed:.0f}s)")
    print(f"Output: {result.output_path}")
    print(f"Scenes: {len(result.scenes)}")
else:
    print(f"FAILED: {result.error}")
print("="*50)

In [ ]:
#@title **CELL 3: Watch Video + Download** { display-mode: "form" }

from IPython.display import HTML, display
from base64 import b64encode
from google.colab import files

if result.status.value == "completed":
    with open(result.output_path, "rb") as f:
        video_data = b64encode(f.read()).decode()

    display(HTML(f'''
    <video width="900" controls autoplay>
        <source src="data:video/mp4;base64,{video_data}" type="video/mp4">
    </video>
    '''))

    print("\nDownloading...")
    files.download(result.output_path)
else:
    print("No video - generation failed. Check Cell 2 output for errors.")

In [ ]:
#@title **CELL 4 (Optional): Preview All Scene Images** { display-mode: "form" }

from IPython.display import display
from PIL import Image

if result.status.value == "completed":
    for i, scene in enumerate(result.scenes):
        print(f"\n--- Scene {i+1} [{scene['emotion']}] [{scene['motion']}] ---")
        print(f"Narration: {scene['narration'][:80]}...")
        print(f"Prompt: {scene['image_prompt'][:100]}...")
        if scene.get('image_path'):
            img = Image.open(scene['image_path'])
            display(img.resize((640, 360)))